### What one row means to me ?
One row represents one content page aggregated over a single month. It summarizes that page's search visibility, click performance, engagement, and content metadata for that month
### which table  ?
fact_content_daily_performance + dim_content
### time window
I am looking at month 2026-03
### what am i ranking 
I am ranking pages by their review opportunity. Pages with high visibility but relatively poor click-through rate and/or engagement receive higher priority for review.(ctr gap later if possible)
### what am i excluding 
I exclude pages with fewer than 500 monthly impressions because CTR calculated from very small numbers is unstable and can fluctuate due to chance, making those pages unreliable for prioritization.


In [12]:
import duckdb
import os, getpass


con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token: ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# The path to the daily performance table
FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"



In [13]:
diagnostic_query = """
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

print(con.sql(diagnostic_query).df())


   total_rows  unique_content_ids
0      519606              519606


In [15]:
query = f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks)      AS total_clicks,
        AVG(gsc_avg_position) AS average_position
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY content_hash_id
"""
print(con.sql(query).df().head())


            content_hash_id  total_impressions  total_clicks  average_position
0  content_74112b22d00bcc27                0.0           0.0               NaN
1  content_6ceff627345edfc9                0.0           0.0               NaN
2  content_e9c767b8d3a18e23                0.0           0.0               NaN
3  content_244cd6cfbd4e367c                0.0           0.0               NaN
4  content_bba530bf441f445f                0.0           0.0               NaN


In [16]:
describe_query = f"""
    DESCRIBE SELECT * FROM {FACT_DAILY} LIMIT 1
"""
print(con.sql(describe_query).df())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [17]:
# 1. The "Before" Query (No filters)
before_query = f"""
    SELECT COUNT(*) AS rows_before
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
"""

# 2. The "After" Query (Filtering for IS TRUE)
after_query = f"""
    SELECT COUNT(*) AS rows_after
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
"""

# Print both so you can compare!
print("--- Row Count Before Filter ---")
print(con.sql(before_query).df())

print("\n--- Row Count After Filter ---")
print(con.sql(after_query).df())


--- Row Count Before Filter ---
   rows_before
0      9841378

--- Row Count After Filter ---
   rows_after
0      364347


In [18]:
client_diagnostic_query = f"""
    SELECT 
        COUNT(DISTINCT client_hash_id) AS clients_with_ga4_in_march
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND ga4_data_available IS TRUE
"""

print(con.sql(client_diagnostic_query).df())


   clients_with_ga4_in_march
0                         41


In [19]:
master_diagnostic = f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT CASE WHEN gsc_data_available IS TRUE THEN client_hash_id END) AS gsc_clients,
    COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE THEN client_hash_id END) AS ga4_clients,
    COUNT(DISTINCT CASE
        WHEN gsc_data_available IS TRUE
         AND ga4_data_available IS TRUE
        THEN client_hash_id
    END) AS both_clients
FROM {FACT_DAILY}
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

print(con.sql(master_diagnostic).df())


   total_clients  gsc_clients  ga4_clients  both_clients
0             55           47           41            34
